# model.py

In [1]:
# %% Dependencies
import torch 
import torch.distributions as td
import torch.nn as nn 
from torch_geometric.utils import dense_to_sparse

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import get_edge_type

# %%
class ContNodeFeats(nn.Module):
    def __init__(self, node_size, cont_node_feat):
        super(ContNodeFeats, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.cont_node_feat_1 = nn.Linear(self.node_size, 
            self.cont_node_feat * self.node_size)

    def forward(self):
        Z = torch.randn(self.node_size)
        X = self.cont_node_feat_1(Z)
        X = X.view(-1, self.cont_node_feat)
        return X

# %%
class DisNodeFeat(nn.Module): 
    def __init__(self, node_size, cell_types):
        super(DisNodeFeat, self).__init__()
        self.node_size = node_size
        self.cell_types = cell_types

        self.cell_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((1, self.cell_types))
            )
        )

    def forward(self):
        self.dist = td.Categorical(logits=self.cell_logits)
        self.sample = self.dist.sample([self.node_size]).squeeze()
        self.logLik = self.dist.log_prob(self.sample).sum()
        return self.sample + 1 # category indexing starts at 1. 

# %%
class AdjacencyMatrix(nn.Module):
    def __init__(self, node_size):
        super(AdjacencyMatrix, self).__init__()
        self.node_size = node_size

        self.edge_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((self.node_size, self.node_size))
            )
        )

    def forward(self):
        self.dist = td.Bernoulli(logits=self.edge_logits)
        self.sample = self.dist.sample().squeeze()
        self.logLik = self.dist.log_prob(self.sample).sum()
        return self.sample

# %% 
class EdgeFeats(nn.Module):
    def __init__(self, node_size, cont_edge_feat):
        super(EdgeFeats, self).__init__() 
        self.node_size = node_size
        self.cont_edge_feat = cont_edge_feat

    def forward(self, A, C_x): 
        # Discrete Edge Features:
        edges = list(zip(A.long()[0], A.long()[1]))
        e_c = list(map(lambda x: get_edge_type(x, C_x.int()), edges))
        e_c = torch.tensor(e_c)
        e_c = e_c.reshape(-1, 1)

        # Continuous Edge Features
        Z = torch.randn(self.node_size**2, 1)
        W = torch.normal(mean=0, std=1, size=(1, self.cont_edge_feat))
        E = Z @ W
        E = E[:A.shape[1]] # match number of edges.

        # Combines continuous and discrete node features.
        edge_features = torch.cat((e_c, E), dim=-1) 
        return edge_features


# %% Full Model 
class EGG(nn.Module):
    def __init__(self, node_size, cont_node_feat, cell_types, cont_edge_feat,): 
        super(EGG, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.cell_types = cell_types
        self.cont_edge_feat = cont_edge_feat

        # Sub-Generator models.
        self.ContNodeFeats = ContNodeFeats(self.node_size, self.cont_node_feat)
        self.DisNodeFeat = DisNodeFeat(self.node_size, self.cell_types)
        self.AdjacencyMatrix = AdjacencyMatrix(self.node_size)
        self.EdgeFeats = EdgeFeats(self.node_size, self.cont_edge_feat)

    def forward(self):
        # Sub-Generator models.
        X = self.ContNodeFeats()

        C_x = self.DisNodeFeat()

        A = self.AdjacencyMatrix()
        A = dense_to_sparse(A)[0]

        E = self.EdgeFeats(A, C_x)
        
        return X, C_x, A, E

# Edit-Distance.py

In [2]:
# %% Edit Distance
from typing import List

import networkx as nx
from networkx import graph_edit_distance

import torch_geometric

import numpy as np 
import multiprocessing as mp 
from multiprocessing import Pool
from functools import partial

sys.path.append("../scripts/ceograph/")
from ceograph import NucleiNet, NucleiData

mp.set_start_method('fork', force=True)

# Helper function for converting Nuclei Data to NetworkX while retaining features.
def nuclei_to_nx(data: NucleiData) -> nx.DiGraph: 

    G = nx.DiGraph()   

    # Add nodes with features
    for i in range(data.num_nodes):
        node_feats = np.hstack((data.cell_type[i].numpy(), data.x[i].numpy()))
        G.add_node(i, node_features=node_feats)

    # Add edges with features
    for i in range(data.num_edges):
        src, tgt = data.edge_index[0, i].item(), data.edge_index[1, i].item()
        G.add_edge(src, tgt, edge_features=data.edge_attr[i].numpy())

    return G


def node_strict_type_match(node_dict_1, node_dict_2): 

    # Quick return false if features names don't match. 
    if not(set(node_dict_1) & set(node_dict_2)):
        return 0

    if node_dict_1['node_features'][0] == node_dict_2['node_features'][0]:
        return 1

    else: 
        return 0 

def single_edit_distance(ob: nx.DiGraph, G: nx.DiGraph,
                         node_match=node_strict_type_match):
    dist = graph_edit_distance(
        ob, G, 
        node_match=node_match, 
        node_del_cost=lambda x: 0, 
        edge_del_cost=lambda x: 0,
        upper_bound=50, 
        timeout=60
    )

    if dist is None: 
        return torch.tensor([50.0])
    else:
        return torch.tensor([dist])

def list_edit_distance(G: nx.DiGraph, obs: List[nx.DiGraph], 
                       dist_fn=single_edit_distance):
    with Pool() as pool: 
        distances = pool.map(partial(dist_fn, G=G), obs)
    
    return torch.stack(distances).mean()


# loss.py

In [3]:
import torch 

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData

# %%
def EGG_loss_3(EGG_Model: EGG, explainee, 
               target, edit_obs, 
               criterion, num_samples=1, 
               lambda_1=1, lambda_2=1, lambda_3=1):

    pred_loss = 0 
    edit_loss = 0

    for sample in range(num_samples):
        for _ in range(100): # max_iter to avoid index out of bounds error. 
            try:
                X, C_x, A, E = EGG_Model()
                example = NucleiData(X, C_x, A, E)
                example = clear_iso_nodes(example).to(torch.device(0))
                explainee_pred = torch.softmax(explainee(example), dim=0).cpu()

                break
        
            except Exception as e:
                print({e})

        pred_loss += (criterion(explainee_pred, target) 
                      * (1 + EGG_Model.DisNodeFeat.logLik
                           + EGG_Model.AdjacencyMatrix.logLik))

        with torch.no_grad(): # TODO: Consider design of doing within func. 
            example_nx = nuclei_to_nx(example.detach().cpu())
            edit_dist = list_edit_distance(example_nx, edit_obs)

        edit_loss += edit_dist * (EGG_Model.DisNodeFeat.logLik
                                + EGG_Model.AdjacencyMatrix.logLik)

    edge_pen = torch.norm(EGG_Model.AdjacencyMatrix.edge_logits, p=1)

    loss = (lambda_1 * pred_loss / num_samples
          + lambda_2 * edit_loss / num_samples
          + lambda_3 * edge_pen)

    print(f'pred_loss: {lambda_1 * pred_loss / num_samples}' +  
          f'edit_loss: {lambda_2 * edit_loss / num_samples}' + 
          f'edge_pen: {lambda_3 * edge_pen}')

    return loss 

# utils.py

In [4]:
# %% Dependencies
from typing import Optional
import copy 
from torch_geometric.utils import remove_isolated_nodes

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData

# %%
def clear_iso_nodes(example: NucleiData, 
                    num_nodes: Optional[int] = None) -> NucleiData: 
    edge_index, _, mask = (
        remove_isolated_nodes(example.edge_index, num_nodes=num_nodes)
    )
    example_masked = NucleiData(
        x = example.x[mask], 
        edge_index = edge_index, 
        cell_type = example.cell_type[mask], 
        edge_attr = example.edge_attr,
    )

    return example_masked

# train.py

In [7]:
# %% Dependencies
import random 
import time
import pickle

from torch.optim import Adam

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData, load_model

# Load ad obs 
path = "../data/slides/LUDA/ad_train_nx_100.pkl"

with open(path, 'rb') as f:
    ad_train_nx_100 = pickle.load(f)

# %% Training Config

# Reproducibility 
random.seed(0)
torch.manual_seed(0)
device = torch.device(0)

# Model to be explained. 
explainee = load_model(path = "../data/trained/epoch_263.pt",
device=device)

# Model Parameters:
max_nodes = 50
cont_node_feats = 11
cell_types = 5
cont_edge_feat = 2

EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat)

# Training Parameters:
num_epochs = 2
learning_rate = 1e-4
optimizer = Adam(EGG_Model.parameters(), lr=learning_rate)

# %% Training Loop 

for epoch in range(num_epochs): 
    start = time.time()

    optimizer.zero_grad()

    losses = [] # stored for plotting. 

    loss = EGG_loss_3(EGG_Model=EGG_Model, explainee=explainee, 
        target=torch.tensor([1.0, 0.0]), edit_obs=ad_train_nx_100, 
        criterion=torch.nn.BCELoss())
    
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    end = time.time()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.10f}" + 
          f"Time: {end-start:.2f} seconds")

pred_loss: -190.93955993652344edit_loss: -89939.625edge_pen: 273.3856201171875
Epoch [1/2], Loss: -89857.1796875000Time: 150.97 seconds
pred_loss: -0.5509692430496216edit_loss: -90049.1171875edge_pen: 273.36895751953125
Epoch [2/2], Loss: -89776.3046875000Time: 141.47 seconds


# Basic Tests

In [ ]:
# Forward Test
EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat)

X, C_x, A, E = EGG_Model()

example = NucleiData(X, C_x, A, E).to(torch.device(0))
example = clear_iso_nodes(example)
print(example)
explainee.eval()
print(explainee(example))

NucleiData(x=[50, 11], edge_index=[2, 1212], edge_attr=[1212, 3], cell_type=[50])
tensor([6.6616, 3.2744], device='cuda:0', grad_fn=<SliceBackward0>)


In [ ]:
# Edit Distance Test
import pickle

# Load ad obs 
path = "../data/slides/LUDA/ad_train_nx_100.pkl"

with open(path, 'rb') as f:
    ad_train_nx_100 = pickle.load(f)

# Load past gen examples
path = "../data/generated/EGG_Simple_3/gen_ADC_examples.pkl"

with open(path, 'rb') as f:
    gen_ADC_examples = pickle.load(f)

In [ ]:
%%time
with torch.no_grad():
    example = NucleiData(X, C_x, A, E)
    example = clear_iso_nodes(example)
    example_nx = nuclei_to_nx(example)
    new_gen = list_edit_distance(example_nx, ad_train_nx_100)

print(new_gen)

tensor(50.)
CPU times: user 4.17 s, sys: 4.72 s, total: 8.89 s
Wall time: 2min 29s


In [ ]:
%%time
# Loss Test
EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat)

EGG_loss_3(EGG_Model=EGG_Model, explainee=explainee, 
    target=torch.tensor([1.0, 0.0]), edit_obs=ad_train_nx_100, 
    criterion=torch.nn.BCELoss())

CPU times: user 8.11 s, sys: 4.82 s, total: 12.9 s
Wall time: 2min 22s


tensor(-90456.8828, grad_fn=<AddBackward0>)